# Task 5: Idiot Sandwich — TRL DPOTrainer + LoRA Persona Alignment (`align-lora-persona`)

In [ ]:
!nvidia-smi
import os, shutil, pathlib

input_root = pathlib.Path('/kaggle/input')
src_dir = next(input_root.rglob('submission_adapter.py')).parent
work_repo = pathlib.Path('/tmp/repo')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(src_dir, work_repo)
os.chdir(work_repo)
print('Found code dataset at:', src_dir)
print('COMMIT:', (work_repo / 'COMMIT.txt').read_text().strip())


In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers==4.57.6" "datasets>=2.18,<5" "trl==1.16.0" "peft>=0.11.0" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps
import torch, transformers, datasets, trl, peft
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
print('Transformers:', transformers.__version__, 'TRL:', trl.__version__, 'PEFT:', peft.__version__)


In [ ]:
# 1. Generate baseline responses from untuned HuggingFaceTB/SmolLM2-360M-Instruct on persona_style_eval.json
!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path HuggingFaceTB/SmolLM2-360M-Instruct \
  --eval_set_path eval_sets/persona_style_eval.json \
  --output_path /kaggle/working/outputs/lora_persona/persona_base_eval.json \
  --generator_name smollm2-360m-instruct-base \
  --dataset_name persona_style_eval \
  --use_chat_template \
  --batch_size 8

# 2. Train LoRA-DPO persona adapter on ramsay_style_preferences.json (200 train / 20 val)
!PYTHONPATH=. python -m src.align.train_lora_dpo --config configs/lora_persona.yaml

# 3. Generate responses from LoRA-DPO merged checkpoint on persona_style_eval.json
!PYTHONPATH=. python -m src.align.generate \
  --model_name_or_path /kaggle/working/outputs/lora_persona/best_merged \
  --eval_set_path eval_sets/persona_style_eval.json \
  --output_path /kaggle/working/outputs/lora_persona/persona_lora_eval.json \
  --generator_name smollm2-360m-instruct-ramsay-lora-dpo \
  --dataset_name persona_style_eval \
  --use_chat_template \
  --batch_size 8

# 4. Remove best_merged weights after generation so output download stays small (adapters + eval JSONs + metrics remain)
import shutil
shutil.rmtree('/kaggle/working/outputs/lora_persona/best_merged', ignore_errors=True)
shutil.rmtree('/kaggle/working/outputs/lora_persona/trl_checkpoints', ignore_errors=True)
